In [65]:
from bs4 import BeautifulSoup
import re
import os
import pandas as pd


## Getting Players Information

In [69]:
def find_general_info(soup, url):
    info = {
        'player_id': None, 'full_name': None, 'birth_date': None, 'born': None,
        'height': None, 'weight': None, 'nationality': None,
        'position': None, 'shoots': None, 'college': None,
        'high_school': None, 'career_length': None, 'link': url,
    }

    meta = soup.find('div', id='meta')
    if meta is None:
        return info

    h1 = meta.find('h1')
    if h1:
        info['full_name'] = h1.text.strip()

    nat_span = meta.find('span', class_='f-i')
    if nat_span:
        info['nationality'] = nat_span.get_text(strip=True)

    for p in meta.find_all('p'):
        # text = p.get_text(strip=True)
        text = re.sub(r'\s+', ' ', p.get_text(separator=' ', strip=True)).strip()
        try:
            if 'Career Length' in text:
                info['career_length'] = text.replace('Career Length:', '').strip()

            elif 'Position' in text:
                m = re.search(r'Position:\s*(.+)', text)
                if m:
                    pos = m.group(1)
                    # strip off a trailing 'Shoots:' fragment if present on the same line
                    pos = re.split(r'Shoots:', pos)[0].strip(' \u25aa')
                    info['position'] = pos.strip()
                m2 = re.search(r'Shoots:\s*(\w+)', text)
                if m2:
                    info['shoots'] = m2.group(1).strip()

            elif 'College' in text:
                info['college'] = text.replace('College:', '').strip()

            elif 'High School' in text:
                info['high_school'] = text.replace('High School:', '').strip()



            elif 'Born' in text:
                # Grab everything after "Born:" up to " in " (if present) or end of string
                m = re.search(r'Born:\s*(.*?)(?:\s+in\s+.*)?$', text)
                if m:
                    info['birth_date'] = m.group(1).strip()
                m2 = re.search(r'\bin\s+(.*)', text)
                if m2:
                    info['born'] = m2.group(1).strip()


            # e.g. 6-5,205lb(196cm, 92kg)
            elif 'lb' in text or 'kg' in text or 'cm' in text:
                m = re.search(r'\((\d{3})cm', text)
                if m:
                    info['height'] = m.group(1).strip()
                m2 = re.search(r'(\d{2,3})kg', text)
                if m2:
                    info['weight'] = m2.group(1).strip()
        except Exception:
            # don't let one malformed field kill the whole player
            continue

    return info


def find_stats(soup):
    stats = {}
    for div in soup.select(".stats_pullout .p1 > div, .stats_pullout .p2 > div, .stats_pullout .p3 > div"):
        try:
            strong = div.find("strong")
            if strong is None:
                continue
            key = strong.get_text(strip=True)
            ps = div.find_all("p")
            if not ps:
                continue
            value = ps[-1].get_text(strip=True)
            stats[key] = float(value)
        except (ValueError, AttributeError):
            continue
    return stats

In [70]:
# Step 1: List the files and open one

PATH = '../nba_player_html'
data_dir = os.listdir(PATH)
print(f'{len(data_dir)} html have been scraped successfully.')

5416 html have been scraped successfully.


In [71]:
for h in data_dir[:5]:
    print(h)

aubucch01.html
willial06.html
barnhle01.html
morriis01.html
jacksja01.html


In [72]:
results = []
BASE = "https://www.basketball-reference.com"
for html in data_dir:
    p = os.path.join(PATH, html)
    with open(p, 'r', encoding='utf-8') as file:
        content = file.read()
    
    soup = BeautifulSoup(content, 'lxml')
    url = BASE + f'/players/{html[0]}/{html}'
    info = find_general_info(soup, url)
    stats = find_stats(soup)
    info.update(stats)
    results.append(info)


In [83]:
df = pd.DataFrame(results)
df.head()

,player_id,full_name,birth_date,born,height,weight,nationality,position,shoots,college,...,G,PTS,AST,FG%,FT%,WS,TRB,FG3%,eFG%,PER
0,None,Chet Aubuchon,"May 18 , 1916","Gary, Indiana us",178,62,us,Guard,Right,Michigan State,...,30.0,2.2,0.7,25.3,54.3,1.2,NaN,NaN,NaN,NaN
1,None,Alondes Williams,"June 19 , 1999","Milwaukee , Wisconsin us",193,95,us,Shooting Guard,Right,"Colleges: Triton College , Oklahoma , Wake Forest",...,13.0,4.2,1.0,55.6,88.9,0.2,2.1,31.6,63.9,15.1
2,None,Leo Barnhorst,"May 11 , 1924","Indianapolis, Indiana us",193,86,us,Small Forward,Right,Notre Dame,...,344.0,9.4,3.2,36.8,66.5,13.3,5.4,NaN,NaN,14.2
3,None,Isaiah Morris,"April 2 , 1969","Richmond, Virginia us",203,103,us,Small Forward,Right,Arkansas,...,25.0,2.2,0.2,45.6,75.0,0.0,0.5,NaN,45.6,10.6
4,None,Jaren Jackson,"October 27 , 1967","New Orleans, Louisiana us",193,86,us,Shooting Guard,Right,Georgetown,...,431.0,5.5,1.2,39.1,76.4,11.5,1.8,35.3,46.9,10.1


In [56]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5416 entries, 0 to 5415
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   player_id      0 non-null      object 
 1   full_name      5416 non-null   str    
 2   birth_date     5412 non-null   str    
 3   born           5412 non-null   str    
 4   height         5416 non-null   str    
 5   weight         5411 non-null   str    
 6   nationality    5416 non-null   str    
 7   position       5416 non-null   str    
 8   shoots         5416 non-null   str    
 9   college        5015 non-null   str    
 10  high_school    4867 non-null   str    
 11  career_length  4686 non-null   str    
 12  link           5416 non-null   str    
 13  G              5416 non-null   float64
 14  PTS            5416 non-null   float64
 15  AST            5416 non-null   float64
 16  FG%            5381 non-null   float64
 17  FT%            5164 non-null   float64
 18  WS             5415

In [84]:
df.isnull().sum()

player_id        5416
full_name           0
birth_date          4
born                4
height              0
weight              5
nationality         0
position            0
shoots              0
college           401
high_school       549
career_length     730
link                0
G                   0
PTS                 0
AST                 0
FG%                35
FT%               252
WS                  1
TRB               292
FG3%             1665
eFG%             1157
PER               348
dtype: int64

### Add `player_id` column

In [86]:
def add_player_id_col(df):
    df.drop(columns='player_id', inplace=True)
    players_id = df['link'].map(lambda x: x.split('/')[-1].replace('.html', ''))
    df.insert(0, 'player_id', players_id)
    return df

def save_new_df_to_csv(df, path):
    df.to_csv(path) 

In [87]:
# insert player_id column to raw data
path = os.path.join('../Data/raw_data', 'players.csv')
df = add_player_id_col(df)
save_new_df_to_csv(df, path)
df.head()

,player_id,full_name,birth_date,born,height,weight,nationality,position,shoots,college,...,G,PTS,AST,FG%,FT%,WS,TRB,FG3%,eFG%,PER
0,aubucch01,Chet Aubuchon,"May 18 , 1916","Gary, Indiana us",178,62,us,Guard,Right,Michigan State,...,30.0,2.2,0.7,25.3,54.3,1.2,NaN,NaN,NaN,NaN
1,willial06,Alondes Williams,"June 19 , 1999","Milwaukee , Wisconsin us",193,95,us,Shooting Guard,Right,"Colleges: Triton College , Oklahoma , Wake Forest",...,13.0,4.2,1.0,55.6,88.9,0.2,2.1,31.6,63.9,15.1
2,barnhle01,Leo Barnhorst,"May 11 , 1924","Indianapolis, Indiana us",193,86,us,Small Forward,Right,Notre Dame,...,344.0,9.4,3.2,36.8,66.5,13.3,5.4,NaN,NaN,14.2
3,morriis01,Isaiah Morris,"April 2 , 1969","Richmond, Virginia us",203,103,us,Small Forward,Right,Arkansas,...,25.0,2.2,0.2,45.6,75.0,0.0,0.5,NaN,45.6,10.6
4,jacksja01,Jaren Jackson,"October 27 , 1967","New Orleans, Louisiana us",193,86,us,Shooting Guard,Right,Georgetown,...,431.0,5.5,1.2,39.1,76.4,11.5,1.8,35.3,46.9,10.1


In [88]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5416 entries, 0 to 5415
Data columns (total 23 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   player_id      5416 non-null   str    
 1   full_name      5416 non-null   str    
 2   birth_date     5412 non-null   str    
 3   born           5412 non-null   str    
 4   height         5416 non-null   str    
 5   weight         5411 non-null   str    
 6   nationality    5416 non-null   str    
 7   position       5416 non-null   str    
 8   shoots         5416 non-null   str    
 9   college        5015 non-null   str    
 10  high_school    4867 non-null   str    
 11  career_length  4686 non-null   str    
 12  link           5416 non-null   str    
 13  G              5416 non-null   float64
 14  PTS            5416 non-null   float64
 15  AST            5416 non-null   float64
 16  FG%            5381 non-null   float64
 17  FT%            5164 non-null   float64
 18  WS             5415

# Missing Values

In [89]:
df.isna().sum()

player_id           0
full_name           0
birth_date          4
born                4
height              0
weight              5
nationality         0
position            0
shoots              0
college           401
high_school       549
career_length     730
link                0
G                   0
PTS                 0
AST                 0
FG%                35
FT%               252
WS                  1
TRB               292
FG3%             1665
eFG%             1157
PER               348
dtype: int64

## birth_date

In [90]:
df.loc[df['birth_date'].isna()][['player_id', 'link']]

,player_id,link
351,smithjo03,https://www.basketball-reference.com/players/s...
422,leedi01,https://www.basketball-reference.com/players/l...
1209,carusal01,https://www.basketball-reference.com/players/c...
4603,lawalga01,https://www.basketball-reference.com/players/l...
